# 1. Import Libraries

In [80]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# 2. Load data

In [81]:
ratings_df = pd.read_csv(
    "dataset/ratings.dat",
    sep="::",
    engine="python",
    names=["user", "item", "rating", "ts"],
    encoding="latin-1",
)

ratings_df

,user,item,rating,ts
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [82]:
print(ratings_df["user"].nunique())
print(ratings_df["item"].nunique())

6040
3706


In [83]:
movies_df = pd.read_csv(
    "dataset/movies.dat",
    sep="::",
    engine="python",
    names=["item", "title", "genres"],
    encoding="latin-1",
)

movies_df

,item,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [84]:
users_df = pd.read_csv(
    "dataset/users.dat",
    sep="::",
    engine="python",
    names=["user", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)

users_df

,user,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


# 3. Data preparation

## 3.1 Filter out movies that have less than 5 ratings, as well as users who rate less than 5 times

In [85]:
print(ratings_df["user"].value_counts().min())
print(ratings_df["item"].value_counts().min())

20
1


In [86]:
counts = ratings_df["item"].value_counts()

filtered_movies_list = counts[counts >= 5].index.tolist()

print(filtered_movies_list)

[2858, 260, 1196, 1210, 480, 2028, 589, 2571, 1270, 593, 1580, 1198, 608, 2762, 110, 2396, 1197, 527, 1617, 1265, 1097, 2628, 2997, 318, 858, 356, 2716, 296, 1240, 1, 1214, 2916, 457, 3578, 1200, 541, 2987, 1259, 50, 34, 2791, 780, 3175, 1193, 919, 924, 1127, 2355, 1387, 1221, 912, 1036, 1213, 1610, 377, 1291, 2000, 1136, 3114, 1307, 1704, 1721, 1968, 648, 2599, 3793, 32, 2174, 2797, 2291, 2918, 590, 3471, 2959, 1374, 2683, 1394, 592, 1784, 1573, 1304, 3418, 223, 380, 2706, 1225, 1584, 1527, 3481, 1923, 2699, 750, 39, 21, 1393, 2804, 588, 2406, 1220, 733, 1230, 3176, 1961, 3751, 1079, 908, 3408, 1073, 3623, 1356, 3527, 2700, 1219, 1247, 1089, 1544, 1222, 150, 3753, 111, 2710, 2657, 357, 1094, 2985, 1206, 3147, 2640, 2395, 3421, 1517, 1641, 1278, 1252, 367, 1358, 1208, 2100, 1676, 2529, 2321, 2011, 920, 2324, 2012, 1090, 47, 1500, 1370, 1954, 1673, 2194, 2115, 1376, 587, 364, 3253, 3671, 1288, 923, 316, 3160, 1242, 1917, 736, 3255, 1294, 1732, 3361, 1396, 2763, 1258, 2302, 3448, 1653, 1

In [87]:
# Filter to keep only rows where the 'item' is in your list
filtered_df = ratings_df[ratings_df["item"].isin(filtered_movies_list)]

filtered_df

,user,item,rating,ts
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [88]:
print(filtered_df["user"].nunique())
print(filtered_df["item"].nunique())

6040
3416


## 3.2 Re-index movie and user ID

number users starting from 0, and movies starting from 1. movie_id 0 is reserved as a padding value, not a real movie. padding will be used later, where a user's history list is filled with movie_id up to a fixed length, and if a user watches less movie than the length of the list, the rest will be filled with 0. since no movie has movie_id=0, the 0 in the list would not be confused with a movie

In [89]:
item_ids = np.sort(filtered_df["item"].unique())
item_map = {i: k + 1 for k, i in enumerate(item_ids)} 

user_ids = np.sort(filtered_df["user"].unique())
user_map = {i: k for k, i in enumerate(user_ids)}

filtered_df["movie_id"] = filtered_df["item"].map(item_map)
filtered_df["user_id"] = filtered_df["user"].map(user_map)

filtered_df = filtered_df.sort_values(by=["user", "ts"])
filtered_df

,user,item,rating,ts,movie_id,user_id
31,1,3186,4,978300019,2758,0
22,1,1270,5,978300055,1069,0
27,1,1721,4,978300055,1445,0
37,1,1022,5,978300055,862,0
24,1,2340,3,978300103,1979,0
...,...,...,...,...,...,...
1000019,6040,2917,4,997454429,2513,6039
999988,6040,1921,4,997454464,1591,6039
1000172,6040,1784,3,997454464,1486,6039
1000167,6040,161,3,997454486,146,6039


## 3.3 Splitting data

In [90]:
train_df = filtered_df.groupby("user", group_keys=False).apply(
    lambda x: x.iloc[:-2]
)
train_df

,user,item,rating,ts,movie_id,user_id
31,1,3186,4,978300019,2758,0
22,1,1270,5,978300055,1069,0
27,1,1721,4,978300055,1445,0
37,1,1022,5,978300055,862,0
24,1,2340,3,978300103,1979,0
...,...,...,...,...,...,...
1000119,6040,3671,4,997454367,3156,6039
999923,6040,232,5,997454398,214,6039
1000019,6040,2917,4,997454429,2513,6039
999988,6040,1921,4,997454464,1591,6039


In [91]:
val_df = filtered_df.groupby("user", group_keys=False).apply(
    lambda x: x.iloc[-2:-1]
)
val_df

,user,item,rating,ts,movie_id,user_id
34,1,1907,4,978824330,1579,0
87,2,1544,4,978300174,1301,1
191,3,3868,3,978298486,3340,2
237,4,1036,4,978294282,876,3
418,5,1884,3,978246576,1558,4
...,...,...,...,...,...,...
999076,6036,3574,1,956755196,3075,6035
999557,6037,968,4,956801840,817,6036
999741,6038,2700,1,956715051,2310,6037
999815,6039,1204,5,956758029,1005,6038


In [92]:
test_df = filtered_df.groupby("user", group_keys=False).apply(
    lambda x: x.iloc[-1:]
)
test_df

,user,item,rating,ts,movie_id,user_id
25,1,48,5,978824351,47,0
136,2,1917,3,978300174,1587,1
232,3,2081,4,978298504,1749,2
243,4,1954,5,978294282,1624,3
258,5,288,2,978246585,267,4
...,...,...,...,...,...,...
999251,6036,2643,1,956755196,2257,6035
999684,6037,435,3,956801840,403,6036
999731,6038,1183,5,956717204,985,6037
999826,6039,1254,4,956758029,1053,6038


## 3.4 Data transformation

plot a matrix where every row is one user_id, and every column is one movie_id. if the user watched the movie, put a 1 under the movie_id, else 0


In [93]:
import scipy.sparse as sp

num_users = filtered_df["user"].nunique()
num_items = filtered_df["item"].nunique()

train_matrix = sp.csr_matrix(
    (np.ones(len(train_df), dtype=np.float32), (train_df["user_id"].to_numpy(), train_df["movie_id"].to_numpy())),
    shape=(num_users, num_items + 1)
)  

print(train_matrix[65:85, 325:345].toarray())
print("Shape:", train_matrix.shape)

[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

# 4. Model 

## 4.1 Popularity model

In [94]:
# prints a list, where the first item is the padding slot, 
# then second item is movie_id 1, which was watched by 2064 users

item_popularity = np.asarray(train_matrix.sum(0)).ravel()
print(item_popularity[:10])

[   0. 2064.  695.  468.  170.  292.  935.  457.   68.  102.]


In [95]:
val_item  = val_df["movie_id"].to_numpy()
print(val_item)

[1579 1301 3340 ... 2310 1005  146]


In [96]:
val_matrix = sp.csr_matrix(
    (np.ones(len(val_df), dtype=np.float32), (val_df["user_id"].to_numpy(), val_df["movie_id"].to_numpy())),
    shape=(num_users, num_items + 1)
)

seen = train_matrix + val_matrix
print(seen[65:85, 325:345].toarray())

[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [97]:
# give the same score to all users, 
# resulting in a table with 6040 rows (one row per user) and 3417 columns (one per movie), 
# where every row is the same  

popularity_scores = np.tile(item_popularity, (num_users, 1))
print(popularity_scores.shape)
print(popularity_scores)

(6040, 3417)
[[   0. 2064.  695. ...   52.   39.  372.]
 [   0. 2064.  695. ...   52.   39.  372.]
 [   0. 2064.  695. ...   52.   39.  372.]
 ...
 [   0. 2064.  695. ...   52.   39.  372.]
 [   0. 2064.  695. ...   52.   39.  372.]
 [   0. 2064.  695. ...   52.   39.  372.]]


In [98]:
# give every watched movie the worst score (negative infinity), 
# so that it wouldn't be recommended 

popularity_scores = popularity_scores.astype(np.float64)

# turn padding column into -inf also so it won't be recommended
popularity_scores[:, 0] = -np.inf            

rows, cols = seen.nonzero()                 
popularity_scores[rows, cols] = -np.inf
print(popularity_scores)

[[ -inf  -inf  695. ...   52.   39.  372.]
 [ -inf 2064.  695. ...   52.   39.  372.]
 [ -inf 2064.  695. ...   52.   39.  372.]
 ...
 [ -inf 2064.  695. ...   52.   39.  372.]
 [ -inf 2064.  695. ...   52.   39.  372.]
 [ -inf  -inf  695. ...   52.   39.  372.]]


In [99]:
popularity_scores_df = test_df.copy() 

test_scores = []
for _, row in test_df.iterrows():                 
    u = row["user_id"]                               
    i = row["movie_id"]                        
    test_scores.append(popularity_scores[u, i])       

popularity_scores_df["score"] = test_scores
popularity_scores_df = popularity_scores_df[["user_id", "movie_id", "score"]] 
popularity_scores_df

,user_id,movie_id,score
25,0,47,372.0
136,1,1587,1102.0
232,2,1749,1025.0
243,3,1624,1116.0
258,4,267,694.0
...,...,...,...
999251,6035,2257,316.0
999684,6036,403,531.0
999731,6037,985,976.0
999826,6038,1053,452.0


In [100]:
test_ranks = []
for _, row in popularity_scores_df.iterrows(): 
    u = int(row["user_id"])                              
    s = row["score"] 
    test_ranks.append((popularity_scores[u] > s).sum())

# rank shows how many movies have a score higher itself
popularity_scores_df["rank"] = test_ranks
popularity_scores_df

,user_id,movie_id,score,rank
25,0,47,372.0,802
136,1,1587,1102.0,119
232,2,1749,1025.0,165
243,3,1624,1116.0,138
258,4,267,694.0,312
...,...,...,...,...
999251,6035,2257,316.0,457
999684,6036,403,531.0,450
999731,6037,985,976.0,205
999826,6038,1053,452.0,623


In [101]:
sample_df = popularity_scores_df.copy()
sample_df["recall@20"] = sample_df["rank"] < 20

sample_df["ndcg@20"] = np.where(
    sample_df["rank"] < 20,
    1 / np.log2(sample_df["rank"] + 2),
    0.0,
)

sample_df

,user_id,movie_id,score,rank,recall@20,ndcg@20
25,0,47,372.0,802,False,0.00000
136,1,1587,1102.0,119,False,0.00000
232,2,1749,1025.0,165,False,0.00000
243,3,1624,1116.0,138,False,0.00000
258,4,267,694.0,312,False,0.00000
...,...,...,...,...,...,...
999251,6035,2257,316.0,457,False,0.00000
999684,6036,403,531.0,450,False,0.00000
999731,6037,985,976.0,205,False,0.00000
999826,6038,1053,452.0,623,False,0.00000


In [102]:
# look at each user's top K recommendations, 
# if the ranking falls within the top K, true, else false
# find the mean number of times true happens, this is known as Recall@K 

# NDCG is a score that rewards getting the right movie higher in the list (closer to 0), 
# not just any position in the top K 

# for recall and NDCG, both run from 0 to 1, higher is better

def score_model(df, k):     
    recall = (df < k).mean()
    print(f"Recall@{k}:", recall)

    ndcg = np.where(df < k, 1 / np.log2(df + 2), 0).mean()
    print(f"NDCG@{k}:", ndcg)

In [103]:
popularity_ranks = popularity_scores_df["rank"].to_numpy()
K = [10,20]
for k in K: 
    score_model(popularity_ranks,k)

Recall@10: 0.0369205298013245
NDCG@10: 0.018032673467874877
Recall@20: 0.06837748344370861
NDCG@20: 0.025880026490064954


## 4.2 Item-KNN 

In [104]:
# co-occurrence: result is a movie × movie table, 
# where each entry counts how many users watched both movies

co = (train_matrix.T @ train_matrix).toarray()

print(co.shape)
print(co[:10, :10])

(3417, 3417)
[[   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.]
 [   0. 2064.  488.  286.  112.  212.  511.  313.   51.   58.]
 [   0.  488.  695.  151.   54.  118.  230.  163.   43.   49.]
 [   0.  286.  151.  468.   62.  122.  149.  153.   19.   35.]
 [   0.  112.   54.   62.  170.   66.   61.   69.    7.   13.]
 [   0.  212.  118.  122.   66.  292.   93.  121.   14.   30.]
 [   0.  511.  230.  149.   61.   93.  935.  142.   16.   61.]
 [   0.  313.  163.  153.   69.  121.  142.  457.   11.   25.]
 [   0.   51.   43.   19.    7.   14.   16.   11.   68.    6.]
 [   0.   58.   49.   35.   13.   30.   61.   25.    6.  102.]]


In [105]:
# turn raw counts into similarity score, as raw count only shows 
# the number of people who watched both movies, but does not take into account 
# audience size. also need to apply a shrinkage factor, so that movies watched 
# only by 1 person dont have a perfect score (due to the math of cosine similarity)

co_noself = co.copy()
np.fill_diagonal(co_noself, 0)    
print(co_noself[:10, :10])

[[  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
 [  0.   0. 488. 286. 112. 212. 511. 313.  51.  58.]
 [  0. 488.   0. 151.  54. 118. 230. 163.  43.  49.]
 [  0. 286. 151.   0.  62. 122. 149. 153.  19.  35.]
 [  0. 112.  54.  62.   0.  66.  61.  69.   7.  13.]
 [  0. 212. 118. 122.  66.   0.  93. 121.  14.  30.]
 [  0. 511. 230. 149.  61.  93.   0. 142.  16.  61.]
 [  0. 313. 163. 153.  69. 121. 142.   0.  11.  25.]
 [  0.  51.  43.  19.   7.  14.  16.  11.   0.   6.]
 [  0.  58.  49.  35.  13.  30.  61.  25.   6.   0.]]


In [106]:
# similarity(a, b) = shared / (sqrt(popularity a) × sqrt(popularity b) + shrinkage)
# eg. 0.404 is the similarity score between movie 1 and 2 == score between movie 2 and 1
# values around 0.2 to 0.4 are strong links between popular movies.
# movie 1 is most similar to movies 6 and 2 (0.365 and 0.404) 
# and least to 9 and 8 (0.124 and 0.133). 
# if someone has watched movie 1, then movies 2 and 6 will get the 
# biggest boosts in their recommendations.

shrink = 10.0           

normalized_popularity = np.sqrt(item_popularity)
print("Pre-normalisation:",item_popularity)
print("Post-normalisation:", normalized_popularity.round(1))

S = co_noself / (np.outer(normalized_popularity, normalized_popularity) + shrink)
print(S[:10, :10].round(3))

Pre-normalisation: [   0. 2064.  695. ...   52.   39.  372.]
Post-normalisation: [ 0.  45.4 26.4 ...  7.2  6.2 19.3]
[[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.404 0.288 0.186 0.27  0.365 0.319 0.133 0.124]
 [0.    0.404 0.    0.26  0.153 0.256 0.282 0.284 0.189 0.177]
 [0.    0.288 0.26  0.    0.212 0.321 0.222 0.324 0.101 0.153]
 [0.    0.186 0.153 0.212 0.    0.284 0.149 0.239 0.06  0.092]
 [0.    0.27  0.256 0.321 0.284 0.    0.175 0.322 0.093 0.164]
 [0.    0.365 0.282 0.222 0.149 0.175 0.    0.214 0.061 0.191]
 [0.    0.319 0.284 0.324 0.239 0.322 0.214 0.    0.059 0.111]
 [0.    0.133 0.189 0.101 0.06  0.093 0.061 0.059 0.    0.064]
 [0.    0.124 0.177 0.153 0.092 0.164 0.191 0.111 0.064 0.   ]]


In [107]:
# for each movie, keep only its top-k most similar neighbours and 
# set all its other similarities to 0.
# most of the 3,417 similarities in each row are tiny and come from a few 
# random shared viewers. adding up hundreds of tiny noisy values for every 
# watched movie drowns out the strong links.
# k is a setting we tune on the validation set
# each row in S_k is a movie, and the values is how much similarity the movie 
# gives to another movie, eg S_k[4,5] means movie 4 gives movie 5 a similarity of 0.284

k = 100

num_rows = S.shape[0]
S_k = S.copy()           

for row in range(1, num_rows):
    movie_row = S_k[row]
    sorted_row = sorted(movie_row, reverse=True)
    threshold = sorted_row[k - 1]

    S_k[row] = np.where(movie_row >= threshold, movie_row, 0)

print("Shape:", S_k.shape)
print(S_k[:10, :10].round(3))
print("----------------------------------")
print(train_matrix[:10, :10].toarray())

Shape: (3417, 3417)
[[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.404 0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.321 0.    0.324 0.    0.   ]
 [0.    0.    0.    0.    0.    0.284 0.    0.239 0.    0.   ]
 [0.    0.    0.    0.321 0.    0.    0.    0.322 0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.189 0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]]
----------------------------------
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 1. 0. 0. 

In [108]:
# scores are calculated for the movies that a user hasn't watched yet, 
# which is equal to the sum of the similarity values from other movies
# eg, user 0's history = [0, 1, 0, 1, 0], which means they watched movies 1 & 3
# goal is to calculate the score for movies 2 & 4, which they haven't yet watched
# row*col multiplication gives the sum of the similarity scores that a movie receives 
# from the movies that the user has watched
# every row is one user

scores_knn = train_matrix @ S_k
print(scores_knn[:10, :10].round(3))

[[ 0.    18.079  3.129  0.     0.     0.     0.82   0.313  0.     0.   ]
 [ 0.    20.093  0.44   0.     0.     0.    16.43   0.32   0.     0.   ]
 [ 0.    15.198  1.555  0.     0.     0.     2.796  0.     0.     0.   ]
 [ 0.     5.96   0.     0.     0.     0.     1.25   0.     0.     0.   ]
 [ 0.    29.661  0.118  0.     0.098  0.    16.769  0.764  0.     0.   ]
 [ 0.    12.551  3.239  0.     0.269  0.854  1.418  2.512  0.     0.   ]
 [ 0.     8.458  0.     0.     0.     0.     8.657  0.     0.     0.   ]
 [ 0.    16.71   0.649  0.331  1.197  0.284 16.122  3.117  0.     0.236]
 [ 0.    28.578  0.     0.331  0.     0.    11.547  1.285  0.     0.   ]
 [ 0.    68.229 19.206  3.005  0.553  6.239  5.233  9.273  0.169  0.   ]]


In [109]:
# turn the movies that a user has already watched into score = -inf 
# so it won't be recommended 

scores = np.asarray(scores_knn, dtype=np.float64)
rows, cols = train_matrix.nonzero()                 
scores[rows, cols] = -np.inf

scores[:, 0] = -np.inf 

print(scores[:10, :10].round(3))

[[  -inf   -inf  3.129  0.     0.     0.     0.82   0.313  0.     0.   ]
 [  -inf 20.093  0.44   0.     0.     0.    16.43   0.32   0.     0.   ]
 [  -inf 15.198  1.555  0.     0.     0.     2.796  0.     0.     0.   ]
 [  -inf  5.96   0.     0.     0.     0.     1.25   0.     0.     0.   ]
 [  -inf 29.661  0.118  0.     0.098  0.      -inf  0.764  0.     0.   ]
 [  -inf   -inf  3.239  0.     0.269  0.854  1.418  2.512  0.     0.   ]
 [  -inf  8.458  0.     0.     0.     0.      -inf  0.     0.     0.   ]
 [  -inf   -inf  0.649  0.331   -inf  0.284 16.122  3.117  0.     0.236]
 [  -inf   -inf  0.     0.331  0.     0.    11.547  1.285  0.     0.   ]
 [  -inf   -inf   -inf  3.005  0.553  6.239  5.233   -inf  0.169  0.   ]]


In [110]:
knn_scores_df = val_df.copy() 

val_scores = []
for _, row in val_df.iterrows():                 
    u = row["user_id"]                               
    i = row["movie_id"]                        
    val_scores.append(scores[u, i])       

knn_scores_df["score"] = val_scores
knn_scores_df = knn_scores_df[["user_id", "movie_id", "score"]] 
knn_scores_df

,user_id,movie_id,score
34,0,1579,5.798222
87,1,1301,19.222279
191,2,3340,0.000000
237,3,876,9.814115
418,4,1558,1.779697
...,...,...,...
999076,6035,3075,0.422762
999557,6036,817,3.560987
999741,6037,2310,1.091273
999815,6038,1005,12.025081


In [111]:
val_ranks = []
for _, row in knn_scores_df.iterrows(): 
    u = int(row["user_id"])                              
    s = row["score"] 
    val_ranks.append((scores[u] > s).sum())

# rank shows how many movies have a score higher itself
knn_scores_df["rank"] = val_ranks
knn_scores_df

,user_id,movie_id,score,rank
34,0,1579,5.798222,91
87,1,1301,19.222279,37
191,2,3340,0.000000,701
237,3,876,9.814115,3
418,4,1558,1.779697,479
...,...,...,...,...
999076,6035,3075,0.422762,1360
999557,6036,817,3.560987,291
999741,6037,2310,1.091273,183
999815,6038,1005,12.025081,68


In [112]:
count = 0 
testtttt = scores[27]
for score in testtttt:
    if score > 26.014493942260742:
        count+=1 

print(count)

12


In [116]:
knn_scores = knn_scores_df["rank"].to_numpy()
K = [10,20]
for k in K: 
    score_model(knn_scores,k)

Recall@10: 0.06390728476821192
NDCG@10: 0.032862614573288554
Recall@20: 0.10430463576158941
NDCG@20: 0.043018531584930866


tune k, then keep the best validation

In [119]:
def knn_val_rank(k):
    print(f"Item-KNN for top {k} values")
    num_rows = S.shape[0]
    S_k = S.copy()           

    for row in range(1, num_rows):
        movie_row = S_k[row]
        sorted_row = sorted(movie_row, reverse=True)
        threshold = sorted_row[k - 1]

        S_k[row] = np.where(movie_row >= threshold, movie_row, 0)

    scores_knn = train_matrix @ S_k
    scores = np.asarray(scores_knn, dtype=np.float64)
    rows, cols = train_matrix.nonzero()                 
    scores[rows, cols] = -np.inf

    scores[:, 0] = -np.inf 

    knn_scores_df = val_df.copy() 

    val_scores = []
    for _, row in val_df.iterrows():                 
        u = row["user_id"]                               
        i = row["movie_id"]                        
        val_scores.append(scores[u, i])       

    knn_scores_df["score"] = val_scores
    knn_scores_df = knn_scores_df[["user_id", "movie_id", "score"]] 

    val_ranks = []
    for _, row in knn_scores_df.iterrows(): 
        u = int(row["user_id"])                              
        s = row["score"] 
        val_ranks.append((scores[u] > s).sum())

    # rank shows how many movies have a score higher itself
    knn_scores_df["rank"] = val_ranks

    knn_scores = knn_scores_df["rank"].to_numpy()
    K = [10,20]
    for top_k in K: 
        score_model(knn_scores,top_k)

    print("=================================")

In [120]:
k_values = [50,100,150,200]
for k in k_values: 
    knn_val_rank(k)

Item-KNN for top 50 values
Recall@10: 0.06804635761589405
NDCG@10: 0.03448194849586907
Recall@20: 0.11572847682119206
NDCG@20: 0.04643956667393202
Item-KNN for top 100 values
Recall@10: 0.06390728476821192
NDCG@10: 0.032862614573288554
Recall@20: 0.10430463576158941
NDCG@20: 0.043018531584930866
Item-KNN for top 150 values
Recall@10: 0.06407284768211921
NDCG@10: 0.03305385552788082
Recall@20: 0.09983443708609271
NDCG@20: 0.04199704118491187
Item-KNN for top 200 values
Recall@10: 0.06175496688741722
NDCG@10: 0.031457825463457094
Recall@20: 0.09635761589403974
NDCG@20: 0.040097259176664624


In [124]:
new_k_values = [4,5,10,20,30,40,50]
for k in new_k_values: 
    knn_val_rank(k)

Item-KNN for top 4 values
Recall@10: 0.07897350993377483
NDCG@10: 0.03898757339890384
Recall@20: 0.13129139072847681
NDCG@20: 0.052103122057356614
Item-KNN for top 5 values
Recall@10: 0.08327814569536424
NDCG@10: 0.04025731670344227
Recall@20: 0.13625827814569536
NDCG@20: 0.05354895164621536
Item-KNN for top 10 values
Recall@10: 0.08129139072847683
NDCG@10: 0.03900455291875239
Recall@20: 0.13228476821192053
NDCG@20: 0.05177405939395147
Item-KNN for top 20 values
Recall@10: 0.0759933774834437
NDCG@10: 0.036219986294728126
Recall@20: 0.12549668874172185
NDCG@20: 0.04858955052710694
Item-KNN for top 30 values
Recall@10: 0.07019867549668875
NDCG@10: 0.03474205120043414
Recall@20: 0.12102649006622516
NDCG@20: 0.047461088232064526
Item-KNN for top 40 values
Recall@10: 0.06937086092715232
NDCG@10: 0.035628835813540434
Recall@20: 0.1173841059602649
NDCG@20: 0.04767010005625954
Item-KNN for top 50 values
Recall@10: 0.06804635761589405
NDCG@10: 0.03448194849586907
Recall@20: 0.11572847682119206


In [ ]:
k = 5

num_rows = S.shape[0]
S_k = S.copy()           

for row in range(1, num_rows):
    movie_row = S_k[row]
    sorted_row = sorted(movie_row, reverse=True)
    threshold = sorted_row[k - 1]

    S_k[row] = np.where(movie_row >= threshold, movie_row, 0)

scores_test = np.asarray(seen @ S_k, dtype=np.float64)
rows, cols = seen.nonzero()                 
scores_test[rows, cols] = -np.inf

scores_test[:, 0] = -np.inf 

knn_scores_test_df = test_df.copy() 

test_scores = []
for _, row in test_df.iterrows():                 
    u = row["user_id"]                               
    i = row["movie_id"]                        
    test_scores.append(scores_test[u, i])       

knn_scores_test_df["score"] = test_scores
knn_scores_test_df = knn_scores_test_df[["user_id", "movie_id", "score"]] 

test_ranks = []
for _, row in knn_scores_test_df.iterrows(): 
    u = int(row["user_id"])                              
    s = row["score"] 
    test_ranks.append((scores_test[u] > s).sum())

# rank shows how many movies have a score higher itself
knn_scores_test_df["rank"] = test_ranks

knn_scores_test = knn_scores_test_df["rank"].to_numpy()
K = [10,20]
for top_k in K: 
    score_model(knn_scores_test,top_k)

Recall@10: 0.08178807947019867
NDCG@10: 0.04019327588633563
Recall@20: 0.13278145695364238
NDCG@20: 0.05300273581529803
